# ARCHON arch exploration on the JAX stack

**The point of this notebook:** show how few lines an architecture experiment costs
once the trainer is a pure function over a params dict. Three demos:

1. load 5k real rows and train the **exact D47c graph** (scaled) for 100 steps,
2. **widen the shared bottleneck** (the L2 128->256 class of experiment, here 32->64):
   a 1-line spec change,
3. **add a second head** (a WDL 3-logit head off the bottleneck, the D41 rich-net idea):
   ~10 lines, no framework ceremony, optimizer/exporter untouched.

Everything is CPU-only by policy (`JAX_PLATFORMS=cpu`) while the GPU belongs to the
live bullet trainer. Full-scale training would run the same code on GPU later.

Launch: `cd jax && JAX_PLATFORMS=cpu XLA_PYTHON_CLIENT_PREALLOCATE=false \
  uv run --no-project --with jax --with optax --with jupyter -- jupyter lab`


In [ ]:
import os, sys, time
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

import numpy as np
import jax, jax.numpy as jnp

from archon_jax.spec import ArchSpec, TrainConfig, RECORD_DTYPE
from archon_jax.features import map_batch
from archon_jax.model import init_params, forward
from archon_jax.losses import total_loss
from archon_jax.optim import bullet_adamw

CORPUS = "/home/nick/projects/chessengine/data/nnue/hardneg/corpus/corpus.filtered.shuf.bin"
recs = np.fromfile(CORPUS, dtype=RECORD_DTYPE, count=5120)
print(f"{len(recs)} rows, jax {jax.__version__} on {jax.devices()}")

## 1. The baseline: exact D47c graph, scaled to CPU size

`ArchSpec` carries the shape knobs; the graph code is identical at every size
(the full-size layout is pinned by the byte-parity proof against the real
sb240 checkpoint).

In [ ]:
def make_trainer(arch, cfg, extra_loss=None):
    import optax
    opt = bullet_adamw(lambda step: cfg.lr)          # constant LR for demos
    def loss_fn(params, batch):
        value_logit, pol_logits, hl2 = forward(params, arch, batch)
        loss, metrics = total_loss(value_logit, pol_logits, batch["target"],
                                   batch.get("pol_target_id"), cfg.pol_lambda)
        if extra_loss is not None:
            extra, name = extra_loss(params, hl2, batch)
            loss = loss + extra
            metrics = {**metrics, name: extra, "loss": loss}
        return loss, metrics
    @jax.jit
    def step(params, opt_state, batch):
        import optax
        (_, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, batch)
        updates, opt_state = opt.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), opt_state, metrics
    return opt, step

def run(arch, cfg, steps=100, extra_params=None, extra_loss=None, seed=1):
    params = init_params(arch, seed)
    if extra_params: params.update(extra_params)
    opt, step = make_trainer(arch, cfg, extra_loss)
    opt_state = opt.init(params)
    batches = [map_batch(recs[i:i+512], wdl=cfg.wdl, score_multiplier=1.0,
                         scale=cfg.scale, policy=False) for i in range(0, 5120, 512)]
    hist = []
    for s in range(steps):
        b = {k: jnp.asarray(v) for k, v in batches[s % len(batches)].items()}
        params, opt_state, m = step(params, opt_state, b)
        hist.append(float(m["value"]))
    return params, hist

cfg  = TrainConfig(batch_size=512, pol_lambda=0.0)
base = ArchSpec(hl=128, l2=32, l3=16, policy=False)
t0 = time.time(); _, h_base = run(base, cfg); dt = time.time() - t0
print(f"baseline  value MSE: {h_base[0]:.4f} -> {h_base[-1]:.4f}   ({dt:.0f}s)")

## 2. Wider bottleneck: one line

The whole experiment is the spec change — loader, loop, losses, exporter all
follow the shapes. (With bullet this is a Rust const edit + full rebuild +
the quant-layout consumers reviewed by hand.)

In [ ]:
wide = ArchSpec(hl=128, l2=64, l3=16, policy=False)   # <- the experiment
_, h_wide = run(wide, cfg)
print(f"bottleneck 32: {h_base[-1]:.4f}   bottleneck 64: {h_wide[-1]:.4f}")

## 3. A second head off the shared bottleneck: ~10 lines

The D41 rich-net WDL head (3-logit softmax-CE on the game result, additive
lambda) — the exact experiment class the migration is meant to make cheap.
`hl2` (the bucket-selected bottleneck) is already returned by `forward`;
a new head = new params + a loss term. No framework, no registry, no
save-format surgery (export of extra heads = two more dict entries).

In [ ]:
arch = ArchSpec(hl=128, l2=32, l3=16, policy=False)
key = jax.random.PRNGKey(0)
wdl_params = {
    "l3_wdlw": 0.25 * jax.random.normal(key, (arch.l2, 3), jnp.float32),
    "l3_wdlb": jnp.zeros(3, jnp.float32),
}

def wdl_loss(params, hl2, batch, lam=0.1):
    logits = hl2 @ params["l3_wdlw"] + params["l3_wdlb"]
    # Demo shortcut: re-derive the result class from the blended target's
    # nearest {0, .5, 1} — real runs carry recs['result'] through the loader.
    cls = jnp.round(batch["target"] * 2).astype(jnp.int32)
    ce = -jax.nn.log_softmax(logits)[jnp.arange(logits.shape[0]), cls]
    return lam * jnp.mean(ce), "wdl_ce"

_, h_multi = run(arch, cfg, extra_params=wdl_params, extra_loss=wdl_loss)
print(f"value MSE with co-trained WDL head: {h_multi[-1]:.4f} (baseline {h_base[-1]:.4f})")

## What this buys at full scale

- optax schedules/optimizers (warmup-cosine, schedule-free, shampoo, ...) drop in
  on the SAME `bullet_adamw`-compatible pytree.
- held-out evals are `forward` + a memmap (`tools/forward_parity.py` pattern) —
  no trainer rebuild, no GPU tenancy.
- new heads/routing axes are dict entries + loss terms; the quant exporter only
  changes if the DEPLOYED layout changes (spec.quant_layout is 12 lines).

The catch (see the study doc): the GPU-scale training-throughput story is
unproven until a JAX-on-GPU window exists, and export fidelity must stay pinned
by the byte-parity + engine-oracle tools on every change.